# Session 1 · Getting Started with Prompting

Generative AI › Track 1 · Prompt Engineering

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-01_prompt_anatomy.py`

In [1]:
"""Build a well-structured prompt from its six building blocks."""
from dataclasses import dataclass, field

@dataclass
class Prompt:
    role: str = ""
    context: str = ""
    task: str = ""
    constraints: list = field(default_factory=list)
    output_format: str = ""
    examples: list = field(default_factory=list)

    def render(self) -> str:
        parts = []
        if self.role:        parts.append(f"ROLE: You are {self.role}.")
        if self.context:     parts.append(f"CONTEXT: {self.context}")
        if self.task:        parts.append(f"TASK: {self.task}")
        if self.constraints: parts.append("CONSTRAINTS:\n" + "\n".join(f"- {c}" for c in self.constraints))
        if self.output_format: parts.append(f"OUTPUT FORMAT: {self.output_format}")
        for i, (inp, out) in enumerate(self.examples, 1):
            parts.append(f"EXAMPLE {i}:\nInput: {inp}\nOutput: {out}")
        return "\n\n".join(parts)

def score(p: Prompt) -> int:
    """Very rough 'completeness' score: one point per filled block (max 6)."""
    return sum(bool(x) for x in [p.role, p.context, p.task, p.constraints, p.output_format, p.examples])

weak = Prompt(task="Write about climate change.")
strong = Prompt(
    role="a friendly science teacher for grade 8 students in Karachi",
    context="Students have just studied the water cycle and have 10 minutes left in class.",
    task="Explain how climate change affects rainfall in Pakistan.",
    constraints=["Maximum 120 words", "Use one local example", "Avoid jargon"],
    output_format="3 short bullet points followed by one quiz question",
)
for name, p in [("WEAK", weak), ("STRONG", strong)]:
    print(f"===== {name} PROMPT (completeness {score(p)}/6) =====")
    print(p.render())
    print()

===== WEAK PROMPT (completeness 1/6) =====
TASK: Write about climate change.

===== STRONG PROMPT (completeness 5/6) =====
ROLE: You are a friendly science teacher for grade 8 students in Karachi.

CONTEXT: Students have just studied the water cycle and have 10 minutes left in class.

TASK: Explain how climate change affects rainfall in Pakistan.

CONSTRAINTS:
- Maximum 120 words
- Use one local example
- Avoid jargon

OUTPUT FORMAT: 3 short bullet points followed by one quiz question



### Example · `session-01_tokens.py`

In [2]:
"""Rough token estimation: most English text averages ~4 characters per token."""
import re

def estimate_tokens(text: str) -> int:
    words = re.findall(r"\w+|[^\w\s]", text)
    by_chars = len(text) / 4
    by_words = len(words) * 0.75 + sum(1 for w in words if not w.isalnum())
    return round((by_chars + by_words) / 2)

samples = {
    "short question": "What is a prompt?",
    "instruction": "Summarise the attached article in five bullet points for a busy manager.",
    "paragraph": ("Generative AI models predict the next token in a sequence. The quality of the "
                  "answer depends heavily on the clarity, context and constraints in the prompt."),
}
context_window = 8000
for name, text in samples.items():
    t = estimate_tokens(text)
    print(f"{name:15} | chars={len(text):3} | ~tokens={t:3} | % of 8k window={t/context_window:.2%}")

cost_per_1k = 0.0005  # illustrative price in USD per 1,000 input tokens
monthly = 200 * 30 * 1500   # 200 requests/day, 30 days, ~1,500 tokens each
print(f"\nMonthly tokens: {monthly:,}  ->  estimated cost ${monthly/1000*cost_per_1k:.2f}")

short question  | chars= 17 | ~tokens=  4 | % of 8k window=0.05%
instruction     | chars= 72 | ~tokens= 14 | % of 8k window=0.18%
paragraph       | chars=155 | ~tokens= 32 | % of 8k window=0.40%

Monthly tokens: 9,000,000  ->  estimated cost $4.50
